In [3]:
## Mirrors the cellPLM-/scGPT-/Geneformer-/scFoundation-embedding-wrapper
## notebooks: load a dataset, train a small VAE on its raw counts, and write
## the per-cell latent representation to an h5ad keyed by obs_names so
## benchmark.ipynb can align it independently of cell order.
##
## Important conceptual difference from the foundation-model wrappers:
##   - cellPLM / scGPT / Geneformer / scFoundation: load PRETRAINED weights,
##     run a forward pass. No training in this notebook.
##   - scVI:                                       TRAIN a fresh VAE from
##     scratch on this dataset. The model has no pretrained weights to share
##     across datasets. Training takes a few minutes on a modern GPU.

In [4]:
import warnings
warnings.filterwarnings('ignore')

import os
import random
from pathlib import Path

import numpy as np
import torch
import anndata as ad
import scvi

In [5]:
DATA_PATH = Path('../../data/cellPLM/data/gse155468.h5ad')
SEED      = 42

# Latent dim. scVI defaults to 10. Benchmarks often use 30 to give the VAE
# enough capacity to compete with the foundation models (cellPLM=512, scGPT=512,
# geneformer=256/768, scfoundation=3072). Larger isn't always better here —
# the VAE prior penalizes unused dims, and 30 is plenty for 11 broad celltypes.
N_LATENT  = 30

# Batch key for batch-effect correction: gse155468 has 11 donors in obs['orig.ident']
# (8 ATAA + 3 Control). Conditioning the decoder on this lets scVI learn a
# donor-invariant latent space — comparable in spirit to what cellPLM does
# implicitly via its cross-cell attention.
BATCH_KEY = 'orig.ident'

# Training schedule. scVI usually converges in 100-400 epochs on datasets this
# size; max_epochs=None lets the built-in heuristic pick (≈ min(round((20000/n_cells)*400), 400)).
MAX_EPOCHS = None

# Seed every layer scvi-tools touches: torch, numpy, python random, lightning.
scvi.settings.seed = SEED
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

Seed set to 42


In [6]:
# scVI models the raw count distribution directly (NB / ZINB likelihood),
# so we feed the count matrix unchanged — NO normalization, NO log1p, NO HVG
# selection. The model learns its own size-factor accounting per cell.
data = ad.read_h5ad(DATA_PATH)
data.obs_names_make_unique()
print(f'data: {data.shape}, X dtype: {data.X.dtype}, has counts: '
      f'{(data.X.sum() % 1 == 0) if hasattr(data.X, "sum") else "?"}')

# setup_anndata registers which obs/layer fields scVI should read. Calling it
# is mandatory before model construction — without it, SCVI(...) raises.
scvi.model.SCVI.setup_anndata(
    data,
    batch_key=BATCH_KEY,        # learned batch embedding -> donor-invariant latent
    layer=None,                  # use adata.X (which holds raw counts here)
)
print(f'setup_anndata done. {data.n_obs} cells × {data.n_vars} genes, '
      f'batches: {data.obs[BATCH_KEY].nunique()}')

data: (48082, 12382), X dtype: int64, has counts: True
setup_anndata done. 48082 cells × 12382 genes, batches: 11


In [7]:
# Build the VAE. n_layers/n_hidden default (1 / 128) is usually enough; we
# bump n_layers=2 / n_hidden=128 for a slightly richer encoder, which costs
# negligible time on this dataset size.
model = scvi.model.SCVI(
    data,
    n_latent=N_LATENT,
    n_layers=2,
    n_hidden=128,
    dropout_rate=0.1,
    dispersion='gene',           # one dispersion param per gene (vs per-batch)
    gene_likelihood='zinb',      # zero-inflated NB — robust to dropout
)
print(model)

SCVI model with the following parameters: 
n_hidden: 128, n_latent: 30, n_layers: 2, dropout_rate: 0.1, dispersion: gene, gene_likelihood: zinb, 
latent_distribution: normal.
Training status: Not Trained
Model's adata is minified?: False

In [8]:
# Train. Lightning auto-detects the GPU; pass accelerator='cpu' if you want
# to force CPU. Progress bar shows per-epoch ELBO + reconstruction loss.
model.train(
    max_epochs=MAX_EPOCHS,
    early_stopping=True,
    early_stopping_patience=20,
    check_val_every_n_epoch=1,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 166/166: 100%|██████████| 166/166 [15:27<00:00,  5.62s/it, v_num=1, train_loss=4.72e+3]

`Trainer.fit` stopped: `max_epochs=166` reached.


Epoch 166/166: 100%|██████████| 166/166 [15:27<00:00,  5.59s/it, v_num=1, train_loss=4.72e+3]


In [9]:
# Per-cell latent representation. By default this returns the posterior mean
# of q(z|x), i.e. the deterministic encoding (give_mean=True), so re-running
# this cell is reproducible.
embedding = model.get_latent_representation(data).astype(np.float32)
print('embedding shape:', embedding.shape, 'dtype:', embedding.dtype)
assert embedding.shape == (data.n_obs, N_LATENT)

embedding shape: (48082, 30) dtype: float32


In [10]:
# Match the cellPLM-/scGPT-/Geneformer-/scFoundation-embedding-wrapper output
# layout so benchmark.ipynb can align by obs_names: X = (n_cells, d) float32,
# obs_names = original cell IDs, var_names = ['dim_0', 'dim_1', ...].
embedding_adata = ad.AnnData(X=embedding)
embedding_adata.obs_names = data.obs_names
embedding_adata.var_names = [f'dim_{i}' for i in range(embedding.shape[1])]
embedding_adata.write_h5ad('gse155468_embedding.h5ad')
print('wrote gse155468_embedding.h5ad', embedding_adata.shape)

wrote gse155468_embedding.h5ad (48082, 30)
